In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

# 1. Read in the data
housing = pd.read_csv("housing.csv")
housing = housing[housing['ocean_proximity'] != '<ISLAND']

In [11]:
# step 1 - feature engineering
# we create new features that describe ratios instead of just totals
housing["rooms_per_household"] = housing["total_rooms"] / housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"] / housing["total_rooms"]
housing["population_per_household"] = housing["population"] / housing["households"]

In [12]:
# handle categorical data and NaN
housing = pd.get_dummies(housing, columns=['ocean_proximity'], drop_first=True, dtype=int)
housing = housing.dropna() # Enkelt sätt, imputering är bättre men detta funkar för nu

In [13]:
# split into X and y
X = housing.drop("median_house_value", axis=1)
y = housing["median_house_value"]


# split into train/test 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
# step 2 - hyperparameter tuning with RandomizedSearchCV
# we define a "grid" of hyperparameters to search
param_dist = {
    'n_estimators': [100, 200, 300],      # Fler träd är oftast bättre
    'max_features': [0.5, 'sqrt', 1.0],   # Hur många features varje träd får se
    'max_depth': [None, 10, 20, 30],      # Hur djupa träden får vara
    'min_samples_leaf': [1, 2, 4]         # Förhindrar överanpassning
}

forest_reg = RandomForestRegressor(random_state=42)


# RandomizedSearchCV tests random combinations and is faster than GridSearch
print("Startar sökning efter bästa modell")
search = RandomizedSearchCV(
    forest_reg, 
    param_distributions=param_dist,
    n_iter=10, # Tests 10 different combinations
    cv=3,      # 3 way cross-validation
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1  # use all CPU cores
)

search.fit(X_train, y_train)

# step 3 - evaluation
best_model = search.best_estimator_
predictions = best_model.predict(X_test)

final_rmse = root_mean_squared_error(y_test, predictions)

print("\n-------------------------------------------")
print(f"Bästa parametrar: {search.best_params_}")
print(f"Nytt RMSE på testdata: {final_rmse:.2f}")
print("-------------------------------------------")

# show all the feature importances
importances = best_model.feature_importances_
feature_names = X_train.columns
indices = np.argsort(importances)[::-1]

print("\nDe 5 viktigaste variablerna:")
for i in range(5):
    print(f"{feature_names[indices[i]]}: {importances[indices[i]]:.4f}")

Startar sökning efter bästa modell

-------------------------------------------
Bästa parametrar: {'n_estimators': 300, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 30}
Nytt RMSE på testdata: 49086.15
-------------------------------------------

De 5 viktigaste variablerna:
median_income: 0.3537
ocean_proximity_INLAND: 0.1608
population_per_household: 0.1104
bedrooms_per_room: 0.0751
longitude: 0.0714
